# feax4d Quickstart — 4D-printing design pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naruki-Ichihara/feax4d/blob/main/examples/colab_quickstart.ipynb)

End-to-end **4D-printing** pipeline built on [FEAX](https://github.com/Naruki-Ichihara/feax):

1. **Optimise** a two-layer thermal shell so a clamped cantilever *stays flat* under a free-edge load — the cooling-induced bilayer warping cancels the load deflection (density + fibre orientation are the design variables).
2. **Fibre paths** — relax an aligned Swift–Hohenberg stripe field along the optimised per-layer fibre director and extract fibre-path polylines.
3. **G-code** — fill each layer's **non-fibre** region with polymer infill (using the optimised density distribution) and emit Fibrifier (9T Labs) G-code, stacking print laminae per design layer to build thickness.
4. **3D view** — render the printed fibre + polymer toolpaths.

This notebook uses the **general** `feax4d` API: the mesh, boundary conditions and loads are built explicitly (here, a rectangular cantilever). Swap in any feax mesh / BCs / loads for other problems.

Tested on a free Colab runtime (CPU and T4 GPU). First-time setup takes ~5–8 minutes (one-off, dominated by the SuiteSparse build). The pipeline below runs at the same resolution as the `examples/*.py` scripts (a few minutes on a GPU runtime; longer on CPU).

## 1. Install

`feax4d` is installed from GitHub. It builds on **FEAX** (the FE backend), so we first run FEAX's Colab setup script (gmsh, NLopt, BLAS/LAPACK, SuiteSparse) and install FEAX exactly as in the FEAX quickstart, then add `feax4d` and its only extra runtime dependency, `shapely` (polymer-infill region geometry).

`feax4d` is installed with `--no-deps` because its `feax` dependency is already satisfied by the git install above (it is not on PyPI).

In [ ]:
!curl -fsSL https://raw.githubusercontent.com/Naruki-Ichihara/feax/main/scripts/colab_setup.sh | bash
!SUITESPARSE_INCLUDE_DIR=/usr/local/include/suitesparse \
 SUITESPARSE_LIBRARY_DIR=/usr/local/lib \
 pip install -q "feax[cuda13,sksparse] @ git+https://github.com/Naruki-Ichihara/feax.git"
!CMAKE_ARGS="-DBUILD_PBATCH_SOLVE=OFF" pip install -q --no-build-isolation git+https://github.com/johnviljoen/spineax.git
# feax4d extra runtime deps not pulled in above (feax brings scikit-image / meshio / matplotlib / nlopt)
!pip install -q shapely tsp-solver2
!pip install -q --no-deps git+https://github.com/Naruki-Ichihara/feax4d.git

In [ ]:
import jax
import feax as fe
import feax4d

print('JAX version   :', jax.__version__)
print('FEAX version  :', fe.__version__)
print('feax4d version:', feax4d.__version__)
print('Backend       :', jax.default_backend())
print('Devices       :', jax.devices())

## 2. Optimise the stable plate (general API)

We build the problem explicitly with the general API — **mesh**, **boundary conditions** and **loads** are all supplied to `OptimizeConfig`:

* **Mesh**: a rectangular plate `Lx × Ly` (`Nx × Ny` QUAD4), span : width = 2 : 1.
* **BC**: the **right** edge is fully clamped (all 5 DOFs of both variables).
* **Load**: a uniform transverse line load on the free (**left**) edge, via `feax4d.uniform_transverse_load`.

The optimiser tailors the per-layer density and fibre orientation so the one-shot cooling ($\Delta T = -150$ K) warps the bilayer just enough to cancel the load deflection — i.e. the plate **stays flat**:

$$ J(\rho, \theta) \;=\; \frac{\lVert w \rVert^2}{\lVert w_{\text{init}} \rVert^2} \;\longrightarrow\; 0 .$$

To solve a different problem, just swap the `mesh`, `bc_specs` and `load_location_fns` / `surface_load_fns`.

In [ ]:
from pathlib import Path
import jax.numpy as jnp
import feax as fe
import feax4d

# --- Rectangular cantilever geometry ---
Lx, Ly, Nx, Ny = 200e-3, 100e-3, 80, 40      # same as examples/stable_plate.py
mesh = fe.mesh.rectangle_mesh(Nx=Nx, Ny=Ny, domain_x=Lx, domain_y=Ly, ele_type='QUAD4')

tol = 1e-6
right = lambda p: jnp.isclose(p[0], Lx,  atol=tol)   # clamped edge
left  = lambda p: jnp.isclose(p[0], 0.0, atol=tol)   # free (loaded) edge

# --- Boundary conditions: fully clamp the right edge (u,v,w and theta_x,theta_y) ---
bc_specs = [
    fe.DirichletBCSpec(location=right, component='all', value=0.0, variable_index=0),
    fe.DirichletBCSpec(location=right, component='all', value=0.0, variable_index=1),
]

# --- Load: uniform transverse line load on the free (left) edge (5 N/m, -z) ---
load_fn = feax4d.uniform_transverse_load(5.0)

cfg = feax4d.OptimizeConfig(
    mesh=mesh,
    bc_specs=bc_specs,
    load_location_fns=(left,),
    surface_load_fns=[load_fn],
    filter_rho_radius=0.05 * Lx,        # absolute filter radii (m)
    filter_theta_radius=0.05 * Lx,
    delta_t=-150.0,                     # one-shot cooling [K]
    target_fn=None,                     # None => flat target (stay-flat objective)
    max_iter=100,
    output_dir=Path('output') / 'stable_plate',
)
result = feax4d.optimize(cfg)
print('\nstop reason :', result.stop_reason)
print('best obj    : %.4e @ iter %d' % (result.best_obj, result.best_iter))
print('history     :', result.xdmf_path)

In [ ]:
import matplotlib.pyplot as plt

h = result.history
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(h['iter'], h['obj'], 'b-')
ax[0].set_xlabel('iteration'); ax[0].set_ylabel(r'$\|w\|^2 / \|w_{init}\|^2$')
ax[0].set_title('stay-flat objective'); ax[0].grid(alpha=0.3)
ax[1].plot(h['iter'], h['vol'], 'g-')
ax[1].set_xlabel('iteration'); ax[1].set_ylabel('mean fibre fraction')
ax[1].set_title('mean density (diagnostic)'); ax[1].set_ylim(0, 1); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Fibre paths (aligned Swift–Hohenberg)

For each layer we relax an aligned-SH stripe field along the optimised fibre director (masked to the fibre region) and extract the stripe centre-lines as fibre-path polylines. Outputs (SVG / NPZ / VTU / PNG) are written under `output/stable_plate/fibre_paths/`.

In [ ]:
paths = feax4d.generate_fibre_paths(
    xdmf_path=result.xdmf_path,
    stripe_period=3.0e-3,    # fibre / tow spacing [m]
    refine_factor=4,         # mesh refinement for the SH relaxation
    n_steps=600,
    rho_cutoff=0.5,
)
fibre_dir = result.xdmf_path.parent / 'fibre_paths'
print('fibre paths per layer:', {k: len(v) for k, v in paths.items()})

## 4. Polymer-filled G-code

Each design layer becomes a **polymer-infill layer (P)** plus a **fibre layer (F)**. The polymer region comes from the optimised **density distribution** — polymer fills only the *non-fibre* part of each layer. Because the polymer and fibre tile complementary in-plane regions, they are printed **coplanar** (same Z) per lamina (`coplanar=True`, the default). Use `layer_print_layers` to stack several laminae per design layer and build part thickness (the part stays a true bilayer).

In [ ]:
params = feax4d.FibrifierParams()
params.temperature.bed_temperature = 90      # deg C
params.layer_height = 0.15                   # mm

gres = feax4d.fibre_paths_to_gcode(
    fibre_dir,
    params=params,
    polymer_fill=True,            # fill non-fibre regions with polymer (density-limited)
    infill_angle=[0.0, 90.0],     # per-layer polymer scan direction
    infill_pitch=1.0,             # polymer line spacing [mm]
    layer_print_layers=[3, 3],    # print laminae per design layer (thickness)
    connection_threshold=10.0,    # merge fibre path ends within 10 mm
)
print('\ngcode        :', gres['gcode_path'])
print('layers       :', gres['n_layers'])
print('fibre paths  :', gres['n_fiber_paths'])
print('polymer paths:', gres['n_polymer_paths'])
print('total fibre  : %.0f mm' % gres['total_fiber_mm'])

## 5. 3D print-path view (interactive)

Render the assembled print stack in 3D with **plotly** — **drag with the mouse to rotate, scroll to zoom**. Each fibre path is colour-coded per path; the polymer infill is drawn faintly behind it. Polymer and fibre of a lamina sit at the same Z (coplanar); the Z axis is auto-exaggerated so the layers are legible.

In [ ]:
# Interactive 3D — drag to rotate, scroll to zoom (plotly; pre-installed on Colab).
fig = feax4d.plot_print_paths_plotly(gres)
fig.show()

### Peek at the generated G-code

In [ ]:
with open(gres['gcode_path']) as f:
    head = f.readlines()[:30]
print(''.join(head))

## Next steps

- **Other problems**: swap `mesh`, `bc_specs` (or `bc_fn`) and `load_location_fns` / `surface_load_fns` in Section 2 — see [`examples/general_problem.py`](https://github.com/Naruki-Ichihara/feax4d/blob/main/examples/general_problem.py) (a doubly-clamped plate).
- Raise `Nx, Ny, max_iter` (Section 2) and `n_steps, refine_factor` (Section 3) for production-quality results.
- Shape matching instead of stay-flat: pass `target_fn=lambda x, y: ...` to `OptimizeConfig` (returns the desired transverse displacement `w_target(x, y)` at the mesh nodes).
- Build thickness with `layer_print_layers=[n0, n1]` (Section 4); set the optimisation ply thickness to `n_i * layer_height` for mechanical consistency.
- Reload a saved design with `feax4d.load_design('output/stable_plate')` and the run summary with `feax4d.load_summary(...)`.

Repository: <https://github.com/Naruki-Ichihara/feax4d>  ·  FE backend: <https://github.com/Naruki-Ichihara/feax>